# Stack Overflow Survey Persona Attribute Extraction

Extract persona **attributes** from Stack Overflow Developer Survey responses using the same persona schema and output contract as the wiki/Amazon extractors.

Unit of extraction:

```text
one survey respondent -> one profile_text -> one persona
```

This notebook is CPU-only through prompt construction. The model demo can use either an OpenAI-compatible API or a local vLLM server. You do **not** need to upload the Stack Overflow CSV files to Hugging Face unless you want to share them with a cluster job or another user.

Pipeline:
1. Point the notebook at your filtered Stack Overflow CSV files.
2. Load one respondent row and convert non-missing answers into canonical `profile_text`.
3. Load `persona/schema/dimensions.json`.
4. Chunk dimensions by category.
5. Build the Stack Overflow extraction prompt (`build_stackoverflow_prompt`).
6. Optionally run a small API/vLLM demo and parse JSON fields.

## 0. Configuration

Edit `SURVEY_ROOT` if your CSV files live somewhere else. For API usage, set `OPENAI_BASE_URL`, `OPENAI_API_KEY`, and `OPENAI_LLM_MODEL` in the environment before running the API demo cell.

In [ ]:
from __future__ import annotations

import json
import os
import textwrap
import urllib.request
import urllib.error
from collections import OrderedDict
from pathlib import Path
from typing import Any

import pandas as pd

# Repo root: this notebook is expected to run from the MatrAIx repo.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'persona/schema/dimensions.json').exists():
    # If the kernel starts inside persona/human_extraction/notebooks, walk upward.
    for parent in Path.cwd().resolve().parents:
        if (parent / 'persona/schema/dimensions.json').exists():
            REPO_ROOT = parent
            break

DIMENSIONS_JSON = REPO_ROOT / 'persona/schema/dimensions.json'

# Local Stack Overflow survey files. No Hugging Face upload is required.
SURVEY_ROOT = Path(os.environ.get(
    'STACKOVERFLOW_DATA_ROOT',
    'D:/Yilan_Fan/study_new/AI Persona Project/Synthesize Persona/persona/curation/attribute_pool/sources/raw/global/stackoverflow_survey',
))

SURVEY_FILES = {
    2023: SURVEY_ROOT / '2023/results_2023_completeness_60.csv',
    2024: SURVEY_ROOT / '2024/results_2024_completeness_60_attention.csv',
    2025: SURVEY_ROOT / '2025/results_2025_completeness_60.csv',
}

# Precomputed, human-readable column mappings exported from the yearly survey schemas.
# These are used directly instead of regenerating mappings from schema_YYYY.txt each run.
MAPPING_FILES = {
    2023: SURVEY_ROOT / '2023/stackoverflow_column_mapping_2023.csv',
    2024: SURVEY_ROOT / '2024/stackoverflow_column_mapping_2024.csv',
    2025: SURVEY_ROOT / '2025/stackoverflow_column_mapping_2025.csv',
}
REPO_MAPPING_FILE = REPO_ROOT / 'persona/human_extraction/docs/stackoverflow_column_mapping_v1.csv'

MISSING_TOKENS = {'', 'na', 'n/a', 'none', 'nan', 'null', '<na>'}
MAX_PROFILE_CHARS = 24000
MODEL_ID = os.environ.get('OPENAI_LLM_MODEL', 'qwen-persona')

print('REPO_ROOT:', REPO_ROOT)
print('DIMENSIONS_JSON:', DIMENSIONS_JSON, 'exists=', DIMENSIONS_JSON.exists())
print('SURVEY_ROOT:', SURVEY_ROOT)
for year, path in SURVEY_FILES.items():
    print(year, path, 'exists=', path.exists())
for year, path in MAPPING_FILES.items():
    print(year, path, 'exists=', path.exists())
print('REPO_MAPPING_FILE:', REPO_MAPPING_FILE, 'exists=', REPO_MAPPING_FILE.exists())

## 1. Load a Stack Overflow CSV sample

The filtered CSV files already remove respondents with low answer completeness. We only load a small sample here so prompt design stays interactive.

In [ ]:
YEAR = 2025
CSV_PATH = SURVEY_FILES[YEAR]

sample = pd.read_csv(CSV_PATH, nrows=5, dtype=str, keep_default_na=False)
print('year:', YEAR)
print('columns:', len(sample.columns))
print('sample rows:', len(sample))
sample.head(3)

## 2. Assemble one respondent into `profile_text`

For Stack Overflow, the input is already structured survey answers. The canonical `profile_text` keeps every non-missing answer, grouped into broad sections. This is analogous to Amazon's `assemble_profile`, but the evidence is survey fields instead of product reviews.

The raw CSV headers are often cryptic, especially for matrix/ranking columns such as `TechEndorse_1` or `AIToolCurrently partially AI`. This notebook now reads the precomputed mapping files directly:

- `2023/stackoverflow_column_mapping_2023.csv`
- `2024/stackoverflow_column_mapping_2024.csv`
- `2025/stackoverflow_column_mapping_2025.csv`

Those files map each raw CSV column to a readable question/sub-item description.

In [ ]:
def is_present(value: Any) -> bool:
    if value is None:
        return False
    return str(value).strip().lower() not in MISSING_TOKENS

def clean_value(value: Any, max_chars: int = 800) -> str:
    text = str(value).strip().replace('\r\n', '\n').replace('\r', '\n')
    text = '\n'.join(line.strip() for line in text.splitlines() if line.strip())
    if len(text) > max_chars:
        text = text[:max_chars].rstrip() + '...'
    return text

def load_column_mapping(year: int) -> dict[str, dict[str, str]]:
    """Load the precomputed Stack Overflow column mapping for one survey year."""
    mapping_path = MAPPING_FILES[year]
    if mapping_path.exists():
        frame = pd.read_csv(mapping_path, dtype=str, keep_default_na=False)
    elif REPO_MAPPING_FILE.exists():
        frame = pd.read_csv(REPO_MAPPING_FILE, dtype=str, keep_default_na=False)
        frame = frame[frame['year'].astype(str) == str(year)]
    else:
        raise FileNotFoundError(
            f'No mapping file found for {year}: {mapping_path} or {REPO_MAPPING_FILE}'
        )

    required = {'column', 'section', 'matched_qname', 'response_suffix', 'description'}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'Mapping file for {year} is missing columns: {sorted(missing)}')

    return {
        row['column']: {
            'section': row.get('section') or 'Other survey answers',
            'matched_qname': row.get('matched_qname') or '',
            'response_suffix': row.get('response_suffix') or '',
            'description': row.get('description') or row['column'],
        }
        for _, row in frame.iterrows()
    }

COLUMN_MAPPING_BY_YEAR = {
    year: load_column_mapping(year) for year in SURVEY_FILES
}

def describe_column(column: str, year: int) -> str:
    return COLUMN_MAPPING_BY_YEAR.get(year, {}).get(column, {}).get('description') or column

def section_for_column(column: str, year: int) -> str:
    return COLUMN_MAPPING_BY_YEAR.get(year, {}).get(column, {}).get('section') or 'Other survey answers'

def assemble_stackoverflow_profile(row: pd.Series | dict[str, Any], year: int) -> str:
    items_by_section: dict[str, list[tuple[str, str, str]]] = OrderedDict()
    row_dict = dict(row)
    for column, value in row_dict.items():
        if column in {'completeness'} or not is_present(value):
            continue
        section = section_for_column(column, year)
        label = describe_column(column, year)
        items_by_section.setdefault(section, []).append((column, label, clean_value(value)))

    response_id = row_dict.get('ResponseId', 'unknown')
    completeness = row_dict.get('completeness')
    header = [
        f'Stack Overflow Developer Survey respondent profile — year={year}, response_id={response_id}.',
    ]
    if is_present(completeness):
        header.append(f'Answer completeness score: {completeness}.')

    lines = [' '.join(header)]
    for section, items in items_by_section.items():
        lines.append('')
        lines.append(f'## {section}')
        for column, label, value in items:
            if label != column:
                lines.append(f'- {column} — {label}: {value}')
            else:
                lines.append(f'- {column}: {value}')

    return '\n'.join(lines)[:MAX_PROFILE_CHARS]

print('Loaded mapping rows:', {year: len(mapping) for year, mapping in COLUMN_MAPPING_BY_YEAR.items()})
print('\nExample column descriptions:')
for column in ['TechEndorse_1', 'LanguageHaveWorkedWith', 'AIToolCurrently partially AI', 'SOVisitFreq']:
    print(f'- {column}: {describe_column(column, YEAR)}')

mapping_preview = pd.DataFrame([
    {'column': column, **COLUMN_MAPPING_BY_YEAR[YEAR].get(column, {})}
    for column in sample.columns[:20]
])
display(mapping_preview)

profile_text = assemble_stackoverflow_profile(sample.iloc[0], YEAR)
print('\n--- profile_text preview ---')
print(profile_text[:3500])

## 3. Load the persona dimension schema

This uses the same schema as the wiki and Amazon extractors. Dimensions are chunked by category so each prompt contains the full respondent profile plus only one slice of dimensions.

In [ ]:
with open(DIMENSIONS_JSON, encoding='utf-8') as fh:
    schema_doc = json.load(fh)

DIMENSIONS = schema_doc['dimensions']
print('schema:', schema_doc.get('name'))
print('total dimensions:', len(DIMENSIONS))

by_category = OrderedDict()
for dim in DIMENSIONS:
    by_category.setdefault(dim.get('category', 'Uncategorized'), []).append(dim)

print('categories:', len(by_category))
for category, dims in list(by_category.items())[:8]:
    print(f'  {category:34s} {len(dims):3d} dims  e.g. {dims[0]["id"]}')

def cat_chunks(by_cat: OrderedDict[str, list[dict[str, Any]]], per_chunk: int = 50) -> list[list[dict[str, Any]]]:
    chunks = []
    for dims in by_cat.values():
        for i in range(0, len(dims), per_chunk):
            chunks.append(dims[i:i + per_chunk])
    return chunks

CHUNKS = cat_chunks(by_category, per_chunk=50)
print('chunks/profile:', len(CHUNKS))

## 4. Stack Overflow extraction prompt

This prompt keeps the wiki/Amazon output contract unchanged, but changes the evidence model: the model should infer from structured survey answers, technology choices, work context, Stack Overflow behavior, AI attitudes, and any free-text answers.

In [ ]:
def build_stackoverflow_prompt(profile_text: str, dimensions: list[dict[str, Any]]) -> str:
    """Stack Overflow respondent persona-extraction prompt.

    Input is ONE respondent's survey answers. Output is a sparse list of only
    evidence-supported attributes, using the same field object shape as the wiki
    and Amazon extractors.
    """
    lines = [
        'You are building a persona for a single Stack Overflow Developer Survey respondent from their survey answers.',
        '',
        'The input is a structured respondent profile assembled from one survey row. It may contain evidence such as the broad categories below.',
        'These are evidence categories, not required output attributes, and they are not exhaustive across survey years.',
        'Only emit attributes from the DIMENSIONS list when directly or strongly supported by the respondent profile.',
        '- BACKGROUND: age bracket, country, education, employment, and learning history when explicitly answered.',
        '- WORK CONTEXT: developer type, years of coding, organization size, remote work, industry, purchase influence, and job satisfaction.',
        '- TECHNICAL PROFILE: languages, databases, platforms, frameworks, tools, operating systems, and development environments.',
        '- COMMUNITY BEHAVIOR: Stack Overflow visits, account status, participation, community membership, and site activities.',
        '- AI PROFILE: AI tool usage, sentiment, trust, workflow uses, perceived risks, AI agents, and free-text AI comments.',
        '- YEAR-SPECIFIC / OTHER SURVEY ANSWERS: attention checks, survey experience, technology endorsement rankings, workplace knowledge questions, tool counts, career changes, or other fields present in a given year.',
        '',
        'Return ONLY JSON with this shape (no markdown, no commentary):',
        '{"fields": [{"field_id": "<one id from DIMENSIONS below>", '
        '"value": "<one allowed value, copied verbatim>", '
        '"confidence": 0.0, '
        '"evidence": "<short quote copied from the respondent profile>", '
        '"description": "<1-2 sentence description of this respondent for this attribute>", '
        '"assignment_type": "direct"}]}',
        '',
        'assignment_type values (Stack Overflow survey context):',
        '- direct: explicitly answered in a survey field, or a deterministic recoding of an explicit answer (for example, age bracket -> age_bracket, country -> region).',
        '- structured_claim: strongly supported by multiple concrete survey answers with little ambiguity.',
        '- summary_inference: a cautious, low-confidence inference from multiple survey answers. Use sparingly.',
        '- unsupported: not supported by the survey answers.',
        '',
        'Sparse extraction policy:',
        '- Return a sparse list: emit an object ONLY for dimensions that are clearly supported by the survey answers.',
        '- Do NOT try to cover every dimension. Missing attributes are better than weak or invented attributes.',
        '- Omit unsupported dimensions entirely. Do not emit null values and do not emit unsupported placeholder objects.',
        '- Assign a value only when the survey response directly or strongly supports that exact dimension.',
        '- If the evidence is generic, indirect, stereotypical, or only weakly associated with the dimension, omit the dimension.',
        '- If an allowed value is more specific than the survey answer, omit the dimension unless the specificity is explicit. For example, generic "Employed" does not prove "Full-time".',
        '',
        'Rules:',
        '- value MUST be exactly one of that dimension\'s allowed values, copied verbatim.',
        '- Every emitted field MUST include a short evidence quote copied verbatim from the respondent profile.',
        '- Evidence MUST include the original column name plus the readable question/sub-item context and the answer value.',
        '- Bad evidence examples: "10", "8", "Yes", "No", "Employed". These are bare values without the survey question context.',
        '- Good evidence example: "TechEndorse_1 — What attracts you to a technology or causes you to endorse it (most to least important)? | Sub-item: AI integration or AI Agent capabilities: 10".',
        '- Every emitted field MUST include a confidence between 0 and 1. Use high confidence only for direct or strong evidence.',
        '- Prefer direct and structured_claim assignments. Use summary_inference only for non-sensitive attributes backed by multiple concrete survey answers.',
        '- description: 1-2 concrete sentences describing THIS respondent for this attribute using survey evidence only. Do not add lifestyle, personality, or career interpretation beyond the evidence.',
        '- Do not infer personality, worldview, family status, sensitive identity, health, politics, religion, income, or housing from generic developer-survey answers.',
        '- Do not infer missing demographics, gender, sexuality, health, disability, family status, religion, ethnicity, politics, income, housing, or socioeconomic status from country, age, job title, technology stack, or developer role unless explicitly answered.',
        '- Do not infer personality traits, values, hobbies, habits, or relationship attributes from technology choices alone.',
        '- Do not infer generation from broad age buckets unless the bucket maps uniquely to one cohort.',
        '- Do not map generic Employment=Employed to Full-time unless full-time is explicitly stated; omit it if no allowed value matches exactly.',
        '- Treat "None of the above" and "None of these" as valid answers, not missing values.',
        '- When unsure, omit the dimension.',
        '- Return valid JSON only, with no markdown.',
        '',
        'DIMENSIONS (field_id — label — description — allowed values):',
    ]
    for dim in dimensions:
        allowed = ' | '.join(str(value) for value in dim.get('values', [])) or '(free value)'
        desc = str(dim.get('description', '')).strip()
        lines.append(f'- {dim["id"]} — {dim.get("label", dim["id"])} — {desc} — [{allowed}]')
    lines += ['', 'RESPONDENT PROFILE:', profile_text]
    return '\n'.join(lines)

def parse_fields(text: str) -> list[dict[str, Any]]:
    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1:
        return []
    try:
        obj = json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return []
    fields = obj.get('fields') if isinstance(obj, dict) else None
    return fields if isinstance(fields, list) else []

demo_category = 'Demographic: Core' if 'Demographic: Core' in by_category else next(iter(by_category))
demo_dims = by_category[demo_category][:6]
prompt_preview = build_stackoverflow_prompt(profile_text, demo_dims)
print(prompt_preview[:2600])

## 5. Optional API demo

Use this if you are calling Qwen through an OpenAI-compatible API, for example DashScope compatible mode, OpenRouter, SiliconFlow, or a remote vLLM server. Keep `RUN_API_DEMO=False` until your endpoint and key are configured.

In [ ]:
RUN_API_DEMO = False

OPENAI_BASE_URL = os.environ.get('OPENAI_BASE_URL', 'http://localhost:8000/v1')
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'EMPTY')
OPENAI_LLM_MODEL = os.environ.get('OPENAI_LLM_MODEL', MODEL_ID)

def chat_completion(prompt: str, max_tokens: int = 4096) -> str:
    payload = {
        'model': OPENAI_LLM_MODEL,
        'messages': [{'role': 'user', 'content': prompt}],
        'temperature': 0.0,
        'top_p': 1.0,
        'max_tokens': max_tokens,
    }
    request = urllib.request.Request(
        OPENAI_BASE_URL.rstrip('/') + '/chat/completions',
        data=json.dumps(payload).encode('utf-8'),
        headers={
            'Authorization': f'Bearer {OPENAI_API_KEY}',
            'Content-Type': 'application/json',
        },
        method='POST',
    )
    with urllib.request.urlopen(request, timeout=180) as response:
        body = json.loads(response.read().decode('utf-8'))
    return body['choices'][0]['message']['content']

if RUN_API_DEMO:
    content = chat_completion(prompt_preview)
    print(content[:2000])
    fields = parse_fields(content)
    print('parsed fields:', len(fields))
else:
    print('API demo skipped. Set RUN_API_DEMO=True after configuring OPENAI_BASE_URL/API_KEY/MODEL.')

## 6. Optional local vLLM demo

Use this only on a GPU node that can load Qwen3.6-35B-A3B. For clean JSON with Qwen3, `enable_thinking=False` is used in the chat template.

In [ ]:
RUN_VLLM_DEMO = False

if RUN_VLLM_DEMO:
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams

    tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3.6-35B-A3B', trust_remote_code=True)
    llm = LLM(
        model='Qwen/Qwen3.6-35B-A3B',
        dtype='bfloat16',
        gpu_memory_utilization=0.90,
        max_model_len=32768,
        enable_prefix_caching=True,
        trust_remote_code=True,
    )
    sampling_params = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=8192)

    text = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': prompt_preview}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    output = llm.generate([text], sampling_params)[0].outputs[0].text
    print(output[:2000])
    print('parsed fields:', len(parse_fields(output)))
else:
    print('Local vLLM demo skipped. Set RUN_VLLM_DEMO=True on a suitable GPU node.')

## 7. Production notes

You do **not** need to upload Stack Overflow data to Hugging Face for extraction. The extractor only needs a readable CSV path. Uploading to Hugging Face is useful only if:

- multiple users or SLURM jobs need a shared canonical copy,
- the CSV is not available on the cluster filesystem, or
- you want dataset versioning/access control.

For a cluster run, copying the filtered CSV files to shared scratch is usually simpler than creating a Hugging Face dataset.

Recommended production structure mirrors wiki/Amazon:

```text
one respondent row -> assemble_stackoverflow_profile(row, year)
same dimensions.json -> category chunks
same build_stackoverflow_prompt(profile_text, chunk)
same parse_fields output -> one JSONL record per respondent
```

For large runs, shard by `ResponseId` hash or by year + row ranges, and make each output JSONL resumable by skipping already-written `ResponseId`s.